# Qwen3.8-27B-Uncensored - Colab + Cloudflare Tunnel### One-click. No credit card. No port forwarding.**FIRST: Runtime > Change runtime type > T4 GPU****Links:**- Google Colab: https://colab.research.google.com/- Cloudflare (free): https://dash.cloudflare.com/sign-up- Tunnel token: https://one.dash.cloudflare.com/ (Zero Trust > Networks > Tunnels)

In [ ]:
# Cell 1: Install!apt-get update -qq && apt-get install -y -qq git-lfs wget curl build-essential cmake git > /dev/null 2>&1!pip install -q gguf mlx-lm > /dev/null 2>&1print('Done')

In [ ]:
# Cell 2: Cloneimport os, glob, time, sys, re, urllib.request, threading, jsonREPO_DIR = '/content/Qwen3.8-27B-Uncensored-MLX'if not os.path.exists(REPO_DIR):    !git lfs install    !git clone https://github.com/onurburak9/Qwen3.8-27B-Uncensored-MLX.git $REPO_DIRfor f in sorted(os.listdir(REPO_DIR)):    fp = os.path.join(REPO_DIR, f)    if os.path.isfile(fp):        sz = os.path.getsize(fp)        print(f'  {f} ({sz/1e6:.1f} MB)' if sz<1e9 else f'  {f} ({sz/1e9:.1f} GB)')    else: print(f'  {f}/')

In [ ]:
# Cell 3: Convert MLX -> GGUF (Q4_K_M)GGUF_DIR = "/content/qwen-gguf"GGUF_FILE = os.path.join(GGUF_DIR, "qwen3.8-27b-uncensored-Q4_K_M.gguf")os.makedirs(GGUF_DIR, exist_ok=True)if os.path.exists(GGUF_FILE):    print(f"GGUF ready: {os.path.getsize(GGUF_FILE)/1e9:.2f} GB")else:    gg = glob.glob(os.path.join(REPO_DIR, "**/*.gguf"), recursive=True)    if gg:        import shutil        shutil.copy2(gg[0], GGUF_FILE)        print(f"Copied: {gg[0]}")    else:        npz = glob.glob(os.path.join(REPO_DIR, "**/*.npz"), recursive=True)        st = glob.glob(os.path.join(REPO_DIR, "**/*.safetensors"), recursive=True)        wts = npz + st        if not wts:            for r, d, fs in os.walk(REPO_DIR):                for fn in fs:                    if fn.endswith((".npz",".safetensors")):                        wts.append(os.path.join(r, fn))        if not wts: raise RuntimeError("No weights found!")        print(f"Found {len(wts)} weight file(s), converting...")        HF_DIR = "/content/qwen-hf-temp"        if not os.path.exists(HF_DIR):            from mlx_lm import convert            convert.convert(hf_path=HF_DIR, mlx_path=REPO_DIR, quantize=False)            print("MLX -> HF done")        LLAMA_DIR = "/content/llama.cpp"        if not os.path.exists(LLAMA_DIR):            !git clone https://github.com/ggerganov/llama.cpp.git $LLAMA_DIR        !python3 $LLAMA_DIR/convert_hf_to_gguf.py $HF_DIR --outfile $GGUF_FILE --outtype q4_k_m        print(f"GGUF created: {os.path.getsize(GGUF_FILE)/1e9:.2f} GB")        import shutil        shutil.rmtree(HF_DIR, ignore_errors=True)

In [ ]:
# Cell 4: Build llama.cpp serverLLAMA_DIR = "/content/llama.cpp"BUILD_DIR = os.path.join(LLAMA_DIR, "build")BIN = os.path.join(BUILD_DIR, "bin", "llama-server")if os.path.exists(BIN):    print("llama-server already built")else:    print("Building llama.cpp with CUDA... (5-10 min)")    !cmake -S $LLAMA_DIR -B $BUILD_DIR -DGGML_CUDA=ON -DLLAMA_CURL=ON -DCMAKE_BUILD_TYPE=Release > /dev/null 2>&1    !cmake --build $BUILD_DIR -j$(nproc) --target llama-server > /dev/null 2>&1    print("Build done!")print(f"Binary: {BIN}")

In [ ]:
# Cell 5: Install cloudflared!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared!chmod +x /usr/local/bin/cloudflaredprint("cloudflared installed")# OPTION A: Quick tunnel (temporary URL, no account needed)# OPTION B: Named tunnel (permanent URL, needs token from Cloudflare)CF_TOKEN = "PASTE_YOUR_TOKEN_HERE"USE_QUICK = CF_TOKEN == "PASTE_YOUR_TOKEN_HERE"if USE_QUICK:    print("Using quick tunnel (temporary URL, changes on reconnect)")    print("For permanent URL, set CF_TOKEN from https://one.dash.cloudflare.com/")else:    print("Named tunnel token configured (persistent URL)")

In [ ]:
# Cell 6: Start llama.cpp serverimport subprocessPORT = 8080BIN = "/content/llama.cpp/build/bin/llama-server"server_proc = subprocess.Popen(    [BIN, "-m", GGUF_FILE, "-c", "8192", "-ngl", "99", "--port", str(PORT),     "--host", "0.0.0.0", "--parallel", "2", "--cont-batching", "-t", "2"],    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)print(f"llama-server PID={server_proc.pid} on port {PORT}")time.sleep(5)if server_proc.poll() is not None:    print("ERROR: Server crashed!")    print(server_proc.stdout.read().decode()[-500:])else:    print("Server starting... model loading takes 2-3 min")

In [ ]:
# Cell 7: Wait for server + start tunnelimport urllib.request, urllib.errorurl = f"http://localhost:{PORT}/v1/models"print("Waiting for server to be ready...")for i in range(120):    try:        urllib.request.urlopen(url, timeout=2)        print(f"Server ready after {(i+1)*2}s")        break    except:        time.sleep(2)else:    raise RuntimeError("Server never became ready! Check Cell 6 output.")tunnel_url = [None]if USE_QUICK:    tunnel_proc = subprocess.Popen(        ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)    def read_quick():        for raw in tunnel_proc.stdout:            line = raw.decode(errors="replace")            if "https://" in line and ".trycloudflare.com" in line:                found = re.search(r"https://([a-z0-9-]+\.trycloudflare\.com)", line)                if found:                    tunnel_url[0] = found.group(1)                    print("")                    print("TUNNEL URL FOUND: https://" + tunnel_url[0])    threading.Thread(target=read_quick, daemon=True).start()    print("Quick tunnel starting... waiting for URL...")else:    tunnel_proc = subprocess.Popen(        ["/usr/local/bin/cloudflared", "tunnel", "run", "--token", CF_TOKEN],        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)    print("Named tunnel starting... check Cloudflare dashboard for URL")    time.sleep(5)    for raw in tunnel_proc.stdout:        line = raw.decode(errors="replace")        if "https://" in line:            print(line.strip())

In [ ]:
# Cell 8: Show URL and testprint("Waiting for tunnel URL...")for i in range(30):    if tunnel_url[0]: break    time.sleep(1)if tunnel_url[0]:    BASE = f"https://{tunnel_url[0]}"    print("")    print("=== COPY THIS INTO YOUR JARVIS .env FILE ===")    print(f"SELF_HOSTED_BASE_URL={BASE}")    print("===========================================")else:    BASE = f"http://localhost:{PORT}"    print(f"No tunnel URL yet. Using localhost: {BASE}")    print("Check Cell 7 output for the tunnel URL.")# Test the endpointimport urllib.request, json as jtry:    payload = j.dumps({        "model": "qwen",        "messages": [{"role": "user", "content": "Say hello in 5 words"}],        "max_tokens": 50    }).encode()    req = urllib.request.Request(        f"{BASE}/v1/chat/completions",        data=payload,        headers={"Content-Type": "application/json"})    resp = urllib.request.urlopen(req, timeout=60)    data = j.loads(resp.read())    msg = data["choices"][0]["message"]["content"]    print(f",Test response: {msg}")    print(",All working! Keep this Colab open.")except Exception as e:    print(f"Test error: {e}")    print("Model may still be loading. Wait 2-3 min and re-run this cell.")

## Done!**Keep this Colab tab open.** If it disconnects, re-run Cells 6-8.Quick tunnel URLs change on reconnect. Named tunnels (with token) stay the same.**Links:**- Google Colab: https://colab.research.google.com/- Cloudflare: https://dash.cloudflare.com/sign-up- Tunnel setup: https://one.dash.cloudflare.com/